# Imports

In [4]:
import numpy as np
from collections import Counter
from tensorflow.keras.datasets import imdb

# Dataset Loading

In [8]:


(X_train_full, y_train_full), (X_test_full, y_test_full) = imdb.load_data()
print(X_train_full[0])  # Print the first review (sequence of word IDs)


[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 22665, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 21631, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 19193, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 10311, 8, 4, 107, 117, 5952, 15, 256, 4, 31050, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 12118, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]


### Dataset Extraction

In [20]:
# 2. Extract balanced training set (2,500 positive, 2,500 negative)
pos_train_idx = np.where(y_train_full == 1)[0][:2500]
neg_train_idx = np.where(y_train_full == 0)[0][:2500]
train_indices = np.concatenate([pos_train_idx, neg_train_idx])

# Shuffle training indices to mix positive and negative samples
np.random.seed(42)
np.random.shuffle(train_indices)

X_train = X_train_full[train_indices]
y_train = y_train_full[train_indices]

# 3. Extract balanced test set (500 positive, 500 negative)
pos_test_idx = np.where(y_test_full == 1)[0][:500]
neg_test_idx = np.where(y_test_full == 0)[0][:500]
test_indices = np.concatenate([pos_test_idx, neg_test_idx])

# Shuffle test indices
np.random.shuffle(test_indices)

X_test = X_test_full[test_indices]
y_test = y_test_full[test_indices]

# 4. Verify subset shapes and class balances
print(f"Train shape: {X_train.shape}, Positive: {np.sum(y_train == 1)}, Negative: {np.sum(y_train == 0)}")
print(f"Test shape:  {X_test.shape}, Positive: {np.sum(y_test == 1)}, Negative: {np.sum(y_test == 0)}")


Train shape: (5000,), Positive: 2500, Negative: 2500
Test shape:  (1000,), Positive: 500, Negative: 500


# Vocab building

In [ ]:
all_ngrams = []
for review in X_train:
    unigrams = [str(token) for token in review]
    bigrams = [f"{unigrams[i]}_{unigrams[i+1]}" for i in range(len(unigrams) - 1)]
    
    # Combine unigrams and bigrams for this review
    review_ngrams = unigrams + bigrams
    all_ngrams.extend(review_ngrams)

ngram_counts = Counter(all_ngrams)

# 3. Select the 2,000 most frequent unigrams and bigrams combined
most_common_2000 = ngram_counts.most_common(2000)
vocab = {ngram: idx for idx, (ngram, count) in enumerate(most_common_2000)}

print(f"Total vocabulary size: {len(vocab)}")
print("Sample vocabulary items:", list(vocab.items())[:10])
print("Most common n-grams:", most_common_2000[:10])

Total vocabulary size: 2000
Sample vocabulary items: [('4', 0), ('5', 1), ('6', 2), ('7', 3), ('8', 4), ('9', 5), ('10', 6), ('11', 7), ('12', 8), ('13', 9)]
Most common n-grams: [('4', 68508), ('5', 33375), ('6', 33311), ('7', 29784), ('8', 27690), ('9', 21836), ('10', 20854), ('11', 19263), ('12', 15963), ('13', 15482)]


# Probabilities calculations

### Freq matrix

In [ ]:
num_reviews = len(X_train)
vocab_size = len(vocab)

# Frequency matrix
X_train_counts = np.zeros((num_reviews, vocab_size), dtype=np.int64)

for idx, review in enumerate(X_train):
    unigrams = [str(token) for token in review]
    bigrams = [f"{unigrams[i]}_{unigrams[i+1]}" for i in range(len(unigrams) - 1)]
    review_ngrams = unigrams + bigrams
    
    for ngram in review_ngrams:
        if ngram in vocab:
            X_train_counts[idx, vocab[ngram]] += 1

### Probabilities

In [ ]:
pos_mask = (y_train == 1)
neg_mask = (y_train == 0)

# Total occurrence count 
count_pos = X_train_counts[pos_mask].sum(axis=0)  
count_neg = X_train_counts[neg_mask].sum(axis=0) 

# Total total words/bigrams across all positive and negative documents
total_words_pos = count_pos.sum()
total_words_neg = count_neg.sum()

# Apply Laplace (Add-1) Smoothing
# V is the vocabulary size (2000)
p_feature_pos = (count_pos + 1) / (total_words_pos + vocab_size)
p_feature_neg = (count_neg + 1) / (total_words_neg + vocab_size)

### Log Ratios

In [ ]:
log_ratios = np.log(p_feature_pos / p_feature_neg)

# Verification
print(f"Computed {len(log_ratios)} log-ratios.")

# Inspect top positive and top negative features
idx_to_vocab = {idx: ngram for ngram, idx in vocab.items()}
sorted_indices = np.argsort(log_ratios)

print("\nTop 5 Strongest Negative Indicators:")
for idx in sorted_indices[:5]:
    print(f"  Feature: '{idx_to_vocab[idx]}' | Log-Ratio: {log_ratios[idx]:.4f}")

print("\nTop 5 Strongest Positive Indicators:")
for idx in sorted_indices[-5:][::-1]:
    print(f"  Feature: '{idx_to_vocab[idx]}' | Log-Ratio: {log_ratios[idx]:.4f}")

Computed 2000 log-ratios.

Top 5 Strongest Negative Indicators:
  Feature: '437_7' | Log-Ratio: -2.7913
  Feature: '437' | Log-Ratio: -2.6259
  Feature: '4_249' | Log-Ratio: -2.3687
  Feature: '862' | Log-Ratio: -2.3206
  Feature: '249' | Log-Ratio: -2.2751

Top 5 Strongest Positive Indicators:
  Feature: '6_389' | Log-Ratio: 1.9552
  Feature: '35_321' | Log-Ratio: 1.8216
  Feature: '1223' | Log-Ratio: 1.8116
  Feature: '61_514' | Log-Ratio: 1.6473
  Feature: '897' | Log-Ratio: 1.6075


In [16]:
p_pos = np.sum(y_train == 1) / len(y_train)
p_neg = np.sum(y_train == 0) / len(y_train)
log_prior_ratio = np.log(p_pos / p_neg)  # Evaluates to 0.0

# Naive_Bayes

In [ ]:
def predict_naive_bayes(X_counts, log_ratios, log_prior_ratio=0.0):
    # Matrix-vector product sums the log-ratios for all features present in each document
    scores = log_prior_ratio + (X_counts @ log_ratios)
    
    # Classify as 1 if score > 0, else 0
    return (scores > 0).astype(int)

# Evaluation

In [ ]:
# Build test count matrix
num_test = len(X_test)
X_test_counts = np.zeros((num_test, len(vocab)), dtype=np.int64)

for idx, review in enumerate(X_test):
    # Using raw ID strings (or decoded words depending on your vocab key format)
    unigrams = [str(token) for token in review]
    bigrams = [f"{unigrams[i]}_{unigrams[i+1]}" for i in range(len(unigrams) - 1)]
    review_ngrams = unigrams + bigrams
    
    for ngram in review_ngrams:
        if ngram in vocab:
            X_test_counts[idx, vocab[ngram]] += 1

y_pred_test = predict_naive_bayes(X_test_counts, log_ratios, log_prior_ratio)

In [ ]:
def compute_metrics(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    
    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return accuracy, precision, recall, f1_score

accuracy, precision, recall, f1 = compute_metrics(y_test, y_pred_test)

print("--- Naive Bayes Baseline Performance ---")
print(f"Accuracy : {accuracy * 100:.2f}%")
print(f"F1-Score : {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")

--- Naive Bayes Baseline Performance ---
Accuracy : 82.20%
F1-Score : 0.8198
Precision: 0.8299
Recall   : 0.8100
